In [ ]:
from google.colab import files
uploaded = files.upload()  # select your zip file when prompted

Saving ucsc-cse-144-spring-2026-final-project.zip to ucsc-cse-144-spring-2026-final-project.zip


In [ ]:
import zipfile
with zipfile.ZipFile("ucsc-cse-144-spring-2026-final-project.zip", 'r') as z:
    z.extractall("data")


In [ ]:
import os, random, numpy as np, torch, torch.nn as nn
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
from PIL import Image
import pandas as pd

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

TRAIN_DIR = "data/train"
TEST_DIR  = "data/test"
BATCH     = 32
EPOCHS    = 15
LR        = 1e-3
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", DEVICE)

# Data transforms
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# Load dataset
full_dataset = ImageFolder(TRAIN_DIR, transform=train_tf)
val_size = int(0.2 * len(full_dataset))
train_size = len(full_dataset) - val_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size],
                                generator=torch.Generator().manual_seed(SEED))
val_ds.dataset.transform = val_tf

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH)
print(f"Train: {train_size}, Val: {val_size}, Classes: {len(full_dataset.classes)}")

# Model: EfficientNet-B0 pretrained
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 100)
model = model.to(DEVICE)

# Optimizer & loss
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
criterion = nn.CrossEntropyLoss()

# Training loop
for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (out.argmax(1) == labels).sum().item()
        total += labels.size(0)

    # Validation
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out = model(imgs)
            val_correct += (out.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    scheduler.step()
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.3f} | "
          f"Train Acc: {100*correct/total:.1f}% | Val Acc: {100*val_correct/val_total:.1f}%")

torch.save(model.state_dict(), "model.pth")
print("Model saved!")


Using: cuda
Train: 864, Val: 215, Classes: 100
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 156MB/s]


Epoch 1/15 | Loss: 3.696 | Train Acc: 18.5% | Val Acc: 30.2%
Epoch 2/15 | Loss: 1.861 | Train Acc: 53.6% | Val Acc: 42.8%
Epoch 3/15 | Loss: 0.947 | Train Acc: 77.9% | Val Acc: 51.2%
Epoch 4/15 | Loss: 0.448 | Train Acc: 90.4% | Val Acc: 52.1%
Epoch 5/15 | Loss: 0.210 | Train Acc: 95.8% | Val Acc: 55.3%
Epoch 6/15 | Loss: 0.100 | Train Acc: 98.5% | Val Acc: 51.2%
Epoch 7/15 | Loss: 0.054 | Train Acc: 99.4% | Val Acc: 54.0%
Epoch 8/15 | Loss: 0.029 | Train Acc: 99.9% | Val Acc: 55.3%
Epoch 9/15 | Loss: 0.027 | Train Acc: 99.9% | Val Acc: 57.2%
Epoch 10/15 | Loss: 0.022 | Train Acc: 99.9% | Val Acc: 56.3%
Epoch 11/15 | Loss: 0.018 | Train Acc: 99.9% | Val Acc: 57.7%
Epoch 12/15 | Loss: 0.014 | Train Acc: 100.0% | Val Acc: 59.1%
Epoch 13/15 | Loss: 0.012 | Train Acc: 99.8% | Val Acc: 58.6%
Epoch 14/15 | Loss: 0.015 | Train Acc: 99.9% | Val Acc: 60.9%
Epoch 15/15 | Loss: 0.013 | Train Acc: 99.9% | Val Acc: 59.5%
Model saved!


In [ ]:
# Generate predictions on test set
model.eval()

test_images = sorted(os.listdir(TEST_DIR), key=lambda x: int(x.split('.')[0]))
predictions = []

with torch.no_grad():
    for img_name in test_images:
        img_path = os.path.join(TEST_DIR, img_name)
        img = Image.open(img_path).convert("RGB")
        img = val_tf(img).unsqueeze(0).to(DEVICE)
        out = model(img)
        pred = out.argmax(1).item()
        predictions.append(pred)

# Make submission file
ids = [int(x.split('.')[0]) for x in test_images]
df = pd.DataFrame({"ID": ids, "Label": predictions})
df = df.sort_values("ID").reset_index(drop=True)
df.to_csv("submission.csv", index=False)
print("Done! First few predictions:")
print(df.head(10))

Done! First few predictions:
   ID  Label
0   0     46
1   1     38
2   2     32
3   3     59
4   4     37
5   5     88
6   6     23
7   7     29
8   8     53
9   9     62


In [ ]:
from google.colab import files
files.download("submission.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Fix: use IDs from sample_submission.csv
sample_df = pd.read_csv("data/sample_submission.csv")
print(sample_df.head(10))
print("Total rows:", len(sample_df))


      ID  Label
0  0.jpg     53
1  1.jpg     43
2  2.jpg     48
3  3.jpg     91
4  4.jpg     72
5  5.jpg     87
6  6.jpg     12
7  7.jpg     33
8  8.jpg     72
9  9.jpg     92
Total rows: 1000


In [ ]:
model.eval()
predictions = []

with torch.no_grad():
    for img_name in sample_df["ID"]:
        img_path = os.path.join(TEST_DIR, img_name)
        img = Image.open(img_path).convert("RGB")
        img = val_tf(img).unsqueeze(0).to(DEVICE)
        out = model(img)
        pred = out.argmax(1).item()
        predictions.append(pred)

sample_df["Label"] = predictions
sample_df.to_csv("submission.csv", index=False)
print("Done!")
print(sample_df.head(10))


Done!
      ID  Label
0  0.jpg     46
1  1.jpg     38
2  2.jpg     32
3  3.jpg     59
4  4.jpg     37
5  5.jpg     88
6  6.jpg     23
7  7.jpg     29
8  8.jpg     53
9  9.jpg     62


In [ ]:
from google.colab import files
files.download("submission.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print("Sample submission rows:", len(sample_df))
print("Test images found:", len(os.listdir(TEST_DIR)))
print("\nSample df tail:")
print(sample_df.tail(10))

Sample submission rows: 1000
Test images found: 1036

Sample df tail:
          ID  Label
990  990.jpg     32
991  991.jpg     70
992  992.jpg     17
993  993.jpg     51
994  994.jpg     51
995  995.jpg     58
996  996.jpg     59
997  997.jpg     91
998  998.jpg     84
999  999.jpg     59


In [ ]:
# Build submission from all actual test images
all_test_images = sorted(os.listdir(TEST_DIR), key=lambda x: int(x.split('.')[0]))
print("Total test images:", len(all_test_images))
print("First few:", all_test_images[:5])
print("Last few:", all_test_images[-5:])


Total test images: 1036
First few: ['0.jpg', '1.jpg', '2.jpg', '3.jpg', '4.jpg']
Last few: ['1031.jpg', '1032.jpg', '1033.jpg', '1034.jpg', '1035.jpg']


In [ ]:
model.eval()
predictions = []

with torch.no_grad():
    for img_name in all_test_images:
        img_path = os.path.join(TEST_DIR, img_name)
        img = Image.open(img_path).convert("RGB")
        img = val_tf(img).unsqueeze(0).to(DEVICE)
        out = model(img)
        pred = out.argmax(1).item()
        predictions.append(pred)

final_df = pd.DataFrame({"ID": all_test_images, "Label": predictions})
final_df.to_csv("submission.csv", index=False)
print("Total rows:", len(final_df))
print(final_df.head(5))

from google.colab import files
files.download("submission.csv")


Total rows: 1036
      ID  Label
0  0.jpg     46
1  1.jpg     38
2  2.jpg     32
3  3.jpg     59
4  4.jpg     37


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print("Class to index mapping (first 10):")
for cls, idx in list(full_dataset.class_to_idx.items())[:10]:
    print(f"  Folder '{cls}' → Label {idx}")

Class to index mapping (first 10):
  Folder '0' → Label 0
  Folder '1' → Label 1
  Folder '10' → Label 2
  Folder '11' → Label 3
  Folder '12' → Label 4
  Folder '13' → Label 5
  Folder '14' → Label 6
  Folder '15' → Label 7
  Folder '16' → Label 8
  Folder '17' → Label 9


In [ ]:
from torchvision.datasets import ImageFolder
import os

# Custom class ordering: 0, 1, 2, ..., 99 (numeric not alphabetic)
class_names = sorted(os.listdir(TRAIN_DIR), key=lambda x: int(x))
class_to_idx = {cls: int(cls) for cls in class_names}

# Rebuild dataset with correct mapping
full_dataset = ImageFolder(TRAIN_DIR, transform=train_tf)
full_dataset.class_to_idx = class_to_idx
full_dataset.targets = [class_to_idx[full_dataset.classes[t]] for t in full_dataset.targets]

# Verify
print("Fixed mapping (first 10):")
for cls, idx in list(full_dataset.class_to_idx.items())[:10]:
    print(f"  Folder '{cls}' → Label {idx}")


Fixed mapping (first 10):
  Folder '0' → Label 0
  Folder '1' → Label 1
  Folder '2' → Label 2
  Folder '3' → Label 3
  Folder '4' → Label 4
  Folder '5' → Label 5
  Folder '6' → Label 6
  Folder '7' → Label 7
  Folder '8' → Label 8
  Folder '9' → Label 9


In [ ]:
# Retrain with fixed labels
val_size = int(0.2 * len(full_dataset))
train_size = len(full_dataset) - val_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size],
                                generator=torch.Generator().manual_seed(SEED))
val_ds.dataset.transform = val_tf

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH)

# Fresh model
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 100)
model = model.to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
criterion = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (model(imgs).argmax(1) == labels).sum().item() if False else correct
        total += labels.size(0)

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            val_correct += (model(imgs).argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    scheduler.step()
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.3f} | Val Acc: {100*val_correct/val_total:.1f}%")

torch.save(model.state_dict(), "model.pth")
print("Done!")


Epoch 1/15 | Loss: 3.753 | Val Acc: 29.8%
Epoch 2/15 | Loss: 1.841 | Val Acc: 43.7%
Epoch 3/15 | Loss: 0.920 | Val Acc: 47.9%
Epoch 4/15 | Loss: 0.438 | Val Acc: 54.0%
Epoch 5/15 | Loss: 0.227 | Val Acc: 51.6%
Epoch 6/15 | Loss: 0.128 | Val Acc: 54.0%
Epoch 7/15 | Loss: 0.041 | Val Acc: 57.7%
Epoch 8/15 | Loss: 0.033 | Val Acc: 57.7%
Epoch 9/15 | Loss: 0.024 | Val Acc: 58.1%
Epoch 10/15 | Loss: 0.016 | Val Acc: 59.1%
Epoch 11/15 | Loss: 0.015 | Val Acc: 60.0%
Epoch 12/15 | Loss: 0.011 | Val Acc: 59.1%
Epoch 13/15 | Loss: 0.013 | Val Acc: 58.1%
Epoch 14/15 | Loss: 0.014 | Val Acc: 59.1%
Epoch 15/15 | Loss: 0.012 | Val Acc: 60.5%
Done!


In [ ]:
model.eval()
predictions = []

with torch.no_grad():
    for img_name in all_test_images:
        img_path = os.path.join(TEST_DIR, img_name)
        img = Image.open(img_path).convert("RGB")
        img = val_tf(img).unsqueeze(0).to(DEVICE)
        out = model(img)
        pred = out.argmax(1).item()
        predictions.append(pred)

final_df = pd.DataFrame({"ID": all_test_images, "Label": predictions})
final_df.to_csv("submission.csv", index=False)
print("Total rows:", len(final_df))

from google.colab import files
files.download("submission.csv")

Total rows: 1036


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os, random, numpy as np, torch, torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset, random_split
from PIL import Image
import pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

TRAIN_DIR = "data/train"
TEST_DIR  = "data/test"
BATCH     = 32
EPOCHS    = 20
LR        = 2e-4
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Correct numeric class ordering
class_names = sorted(os.listdir(TRAIN_DIR), key=lambda x: int(x))

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# Custom dataset with correct numeric label mapping
class CustomDataset(Dataset):
    def __init__(self, root, class_names, transform=None):
        self.samples = []
        self.transform = transform
        for label, cls in enumerate(class_names):
            cls_dir = os.path.join(root, cls)
            for img_file in os.listdir(cls_dir):
                self.samples.append((os.path.join(cls_dir, img_file), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

full_dataset = CustomDataset(TRAIN_DIR, class_names, transform=train_tf)

# Verify
print("Total samples:", len(full_dataset))
print("First 5 samples (path, label):")
for path, label in full_dataset.samples[:5]:
    print(f"  {os.path.basename(os.path.dirname(path))} → Label {label}")

val_size = int(0.2 * len(full_dataset))
train_size = len(full_dataset) - val_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size],
                                generator=torch.Generator().manual_seed(SEED))

# Apply val transform to val set
class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        path, label = self.subset.dataset.samples[self.subset.indices[idx]]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label

val_ds = TransformSubset(train_ds if False else val_ds, val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, num_workers=2)
print(f"Train: {train_size}, Val: {val_size}")

# EfficientNet-B2 (bigger than B0, better accuracy)
model = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 100)
model = model.to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

best_val = 0
for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (out.argmax(1) == labels).sum().item()
        total += labels.size(0)

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            val_correct += (model(imgs).argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    val_acc = 100 * val_correct / val_total
    scheduler.step()

    if val_acc > best_val:
        best_val = val_acc
        torch.save(model.state_dict(), "best_model.pth")

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.3f} | "
          f"Train: {100*correct/total:.1f}% | Val: {val_acc:.1f}% {'★' if val_acc == best_val else ''}")

print(f"\nBest val acc: {best_val:.1f}%")

Total samples: 1079
First 5 samples (path, label):
  0 → Label 0
  0 → Label 0
  0 → Label 0
  0 → Label 0
  0 → Label 0
Train: 864, Val: 215
Downloading: "https://download.pytorch.org/models/efficientnet_b2_rwightman-c35c1473.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b2_rwightman-c35c1473.pth


100%|██████████| 35.2M/35.2M [00:00<00:00, 143MB/s]


Epoch 1/20 | Loss: 4.459 | Train: 8.9% | Val: 12.6% ★
Epoch 2/20 | Loss: 3.804 | Train: 28.2% | Val: 16.7% ★
Epoch 3/20 | Loss: 3.176 | Train: 39.6% | Val: 26.5% ★
Epoch 4/20 | Loss: 2.702 | Train: 48.5% | Val: 34.0% ★
Epoch 5/20 | Loss: 2.362 | Train: 61.1% | Val: 40.5% ★
Epoch 6/20 | Loss: 2.112 | Train: 65.9% | Val: 44.2% ★
Epoch 7/20 | Loss: 1.876 | Train: 74.5% | Val: 46.5% ★
Epoch 8/20 | Loss: 1.719 | Train: 78.9% | Val: 50.2% ★
Epoch 9/20 | Loss: 1.553 | Train: 83.4% | Val: 53.5% ★
Epoch 10/20 | Loss: 1.446 | Train: 86.7% | Val: 54.4% ★
Epoch 11/20 | Loss: 1.365 | Train: 88.7% | Val: 56.3% ★
Epoch 12/20 | Loss: 1.309 | Train: 90.4% | Val: 55.8% 
Epoch 13/20 | Loss: 1.254 | Train: 92.0% | Val: 57.7% ★
Epoch 14/20 | Loss: 1.219 | Train: 92.9% | Val: 57.2% 
Epoch 15/20 | Loss: 1.189 | Train: 93.8% | Val: 57.7% ★
Epoch 16/20 | Loss: 1.183 | Train: 93.6% | Val: 56.7% 
Epoch 17/20 | Loss: 1.172 | Train: 94.6% | Val: 58.1% ★
Epoch 18/20 | Loss: 1.141 | Train: 95.9% | Val: 56.7% 
Epoch 

In [ ]:
# Load best model and generate submission
model.load_state_dict(torch.load("best_model.pth"))
model.eval()
predictions = []

with torch.no_grad():
    for img_name in all_test_images:
        img_path = os.path.join(TEST_DIR, img_name)
        img = Image.open(img_path).convert("RGB")
        img = val_tf(img).unsqueeze(0).to(DEVICE)
        out = model(img)
        pred = out.argmax(1).item()
        predictions.append(pred)

final_df = pd.DataFrame({"ID": all_test_images, "Label": predictions})
final_df.to_csv("submission.csv", index=False)
print("Rows:", len(final_df))

from google.colab import files
files.download("submission.csv")

Rows: 1036


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>